# Préparation du dataset narrateur (version Google Colab)

Ce notebook prépare le dataset de fine-tune du narrateur **sur Colab** : transcription Whisper (large-v3 sur GPU T4),
découpe en clips de 3-15 s, filtre qualité, split 90/10 train/dev, export d'un `dataset_narrateur.zip`.

## Avant de lancer

1. Mets tes fichiers audio (WAV en direct **ou un ZIP qui les contient**, le nom du zip n'a pas d'importance)
   dans ton Google Drive, à cet emplacement exact : **`Mon Drive/thierry_blanc/`**
2. Dans Colab : **Exécution → Modifier le type d'exécution → T4 GPU**
3. **Exécution → Tout exécuter** (et autorise l'accès au Drive quand il le demande)

À la fin, le `dataset_narrateur.zip` est copié dans **`Mon Drive/omnivoice/`** : il est prêt pour l'entraînement
(notebook 2/2), sur Kaggle où ailleurs.

In [ ]:
# ===== CONFIGURATION =====
SIM_THRESHOLD = 0.75  # similarité vocale minimale avec le clip narrateur (section 6)
SPLIT_SEED = 42       # seed du découpage train/dev : même valeur => mêmes clips dans chaque set à chaque régénération
DEV_RATIO = 0.10      # part des clips réservés au dev (0.15 si tu veux un test plus robuste)

import torch
HAS_GPU = torch.cuda.is_available()
# Session CPU : bascule automatique sur Whisper medium en int8 (2 a 4x plus lent, mais correct).
# Session GPU : Whisper large-v3 en float16, comme avant.
WHISPER_SIZE = "large-v3" if HAS_GPU else "medium"
DEVICE = "cuda" if HAS_GPU else "cpu"
COMPUTE = "float16" if HAS_GPU else "int8"
print("Accélérateur :", torch.cuda.get_device_name(0) if HAS_GPU else "aucun GPU -> Whisper " + WHISPER_SIZE + " sur CPU")

## 0. Montage du Google Drive

In [ ]:
import os
os.environ["HF_TOKEN"] = "hf_oceFShVegENoBtXsCrFzfDSXKvnxnJS"

In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")

DRIVE_INPUT = "/content/drive/MyDrive/thierry_blanc"   # dossier Drive contenant tes WAV/ZIP
DRIVE_OUTPUT = "/content/drive/MyDrive/omnivoice"     # où le dataset final sera copié
WORK = "/content/work"

os.makedirs(WORK, exist_ok=True)
assert os.path.isdir(DRIVE_INPUT), (
    "Dossier Drive introuvable : " + DRIVE_INPUT + "\n"
    "Crée le dossier 'thierry_blanc' à la racine de ton Drive et déposes-y tes WAV (ou un ZIP les contenant)."
)
print("Drive monté. Contenu de", DRIVE_INPUT, ":")
for p in sorted(os.listdir(DRIVE_INPUT))[:20]:
    print("  -", p)

## 1. Installation (~2 min)

In [ ]:
%pip install -q faster-whisper "soundfile>=0.12"
print("Installation OK")

from google.colab import userdata
userdata.get('HF_TOKEN')

## 2. (Optionnel, pipeline complet) Télécharger les épisodes
Colle les liens dans `EPISODE_URLS` (une chaîne par épisode). yt-dlp gère la plupart des sites.
Sinon, ignore cette section (mode upload). Réservé à un usage privé.

In [ ]:
import os, subprocess, sys

EPISODE_URLS = []  # <-- colle ici tes liens, ex: ["https://.../ep25.html", "https://.../ep26.html"]

if not EPISODE_URLS:
    print("EPISODE_URLS vide : rien à télécharger (ignore cette section en mode rapide).")
else:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "yt-dlp"], check=True)
    os.makedirs("/content/work/episodes", exist_ok=True)
    for i, url in enumerate(EPISODE_URLS, 1):
        print("Téléchargement :", url)
        subprocess.run(
            [
                "yt-dlp", "-x", "--audio-format", "wav", "--audio-quality", "0",
                "-o", f"/content/work/episodes/ep_{i:02d}.%(ext)s", url,
            ],
            check=True,
        )
    print("Épisodes téléchargés dans /content/work/episodes")

In [ ]:
import os, glob, zipfile

INPUT_DIR = DRIVE_INPUT  # dossier Drive monté (section 0)

# Extraction automatique des éventuels ZIP uploadés
for z in glob.glob(f"{INPUT_DIR}/**/*.zip", recursive=True):
    dest = os.path.join("/content/work/audio_zips", os.path.splitext(os.path.basename(z))[0])
    os.makedirs(dest, exist_ok=True)
    with zipfile.ZipFile(z) as zf:
        zf.extractall(dest)
    print("ZIP extrait :", z)

PATTERNS = ["wav", "mp3", "flac", "m4a", "ogg"]
FILES = sorted(set(
    p
    for pat in PATTERNS
    for p in glob.glob(f"{INPUT_DIR}/**/*.{pat}", recursive=True)
    + glob.glob("/content/work/audio_zips/**/*.{pat}", recursive=True)
    + glob.glob(f"/content/work/episodes/*.{pat}")
))
print(f"{len(FILES)} fichier(s) audio trouvé(s) :")
for f in FILES[:20]:
    print("  -", f)
if not FILES:
    raise RuntimeError(
        "Aucun fichier audio.\n"
        "Vérifie que tes WAV/ZIP sont bien dans Mon Drive/thierry_blanc/, ou remplis EPISODE_URLS à la section 3, puis relance."
    )

## 4. (Optionnel, pipeline complet) Enlever la musique de fond (Demucs)
Démêle voix / musique-effets avec `htdemucs_ft`. Seule la piste `vocals` est conservée.
`SEPARATE_MUSIC = False` si tes fichiers ne contiennent déjà que la voix.
Les fichiers plus longs que 10 minutes sont automatiquement découpés en tranches (un fichier d'une heure saturerait la RAM de Kaggle et serait tué par le système).

In [ ]:
import os, gc, glob, sys, subprocess
import soundfile as sf

SEPARATE_MUSIC = False  # voix d'audiolivre déjà isolée ; remets True s'il y a de la musique de fond à séparer
CHUNK_MIN = 10         # les fichiers plus longs sont découpés en tranches de N minutes
                       # (un fichier d'une heure provoque un SIGKILL : Demucs sature la RAM)
FILES = globals().get("FILES", [])

if not SEPARATE_MUSIC:
    print("Séparation désactivée (audio déjà isolé)")
elif not FILES:
    print("Aucun fichier à séparer")
else:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "demucs"], check=True)
    os.makedirs("/content/work/separated", exist_ok=True)

    # --- Pré-découpage des fichiers longs (lecture par blocs : très faible empreinte RAM) ---
    IN_DIR = "/content/work/demucs_in"
    os.makedirs(IN_DIR, exist_ok=True)
    to_process = []
    for f in FILES:
        try:
            info = sf.info(f)
        except Exception:
            to_process.append(f)  # pas lisible par soundfile : Demucs s'en chargera directement
            continue
        if info.duration <= CHUNK_MIN * 60 + 60:
            to_process.append(f)
            continue
        print(f"Découpe de {f} ({info.duration / 60:.0f} min) en tranches de {CHUNK_MIN} min")
        base = os.path.splitext(os.path.basename(f))[0]
        with sf.SoundFile(f) as fh:
            sr = fh.samplerate
            frames_per_chunk = CHUNK_MIN * 60 * sr
            idx = 0
            while True:
                data = fh.read(frames_per_chunk, dtype="float32", always_2d=True)
                if len(data) == 0:
                    break
                out = os.path.join(IN_DIR, f"{base}__{idx:03d}.wav")
                sf.write(out, data.T, sr)
                to_process.append(out)
                idx += 1
                if len(data) < frames_per_chunk:
                    break
    gc.collect()
    print(len(to_process), "morceaux à séparer")

    for f in to_process:
        print("Séparation de :", os.path.basename(f))
        subprocess.run(
            [
                "demucs", "--two-stems", "vocals", "-n", "htdemucs_ft",
                "--device", "cuda" if globals().get("HAS_GPU", True) else "cpu",
                "-o", "/content/work/separated", f,
            ],
            check=True,
        )
    FILES = sorted(glob.glob("/content/work/separated/htdemucs_ft/*/vocals.wav"))
    print("Voix isolées (" + str(len(FILES)) + " fichiers) :")
    for f in FILES:
        print("  -", f)

## 5. Transcription + découpe (~5-10 min)
faster-whisper transcrit en français et découpe des clips de 3 à 15 s (mono 24 kHz),
avec un score de qualité par clip. À la fin, les 10 plus longs clips s'affichent en lecteurs audio :
repère l'ID d'un clip 100 % narrateur pour la section 6 (inutile si ton audio ne contient que ta voix).

In [ ]:
import ctypes, glob, os, random
import numpy as np
import soundfile as sf
import torch, torchaudio
import IPython.display as ipd
from faster_whisper import WhisperModel

# Précharger libcublas/libcudnn installés par pip avant que CTranslate2 ne les cherche
if torch.cuda.is_available():
    import nvidia
    def _find_cuda_libs():
        libs = []
        for base in getattr(nvidia, "__path__", []):
            libs.extend(glob.glob(f"{base}/*/lib/libcublas*.so*"))
            libs.extend(glob.glob(f"{base}/*/lib/libcudnn*.so*"))
        return libs
    _libs = _find_cuda_libs()
    if not _libs:
        import subprocess, sys
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "nvidia-cublas-cu12", "nvidia-cudnn-cu12"])
        _libs = _find_cuda_libs()
    assert _libs, "libcublas introuvable : relance avec un GPU T4/P100 et Internet ON"
    for _lib in sorted(_libs):
        ctypes.CDLL(_lib, mode=ctypes.RTLD_GLOBAL)
    print("Bibliothèques CUDA préchargées :", len(_libs))

WORK = "/content/work"
random.seed(42)
TARGET_SR = 22050   # fréquence des clips découpés (format natif Piper)
WHISPER_SR = 16000  # fréquence attendue par Whisper
MIN_CLIP, MAX_CLIP = 3.0, 14.5
SILENCE_DB = -38.0  # seuil de silence relatif au pic de l'épisode (dB)
MIN_SILENCE_S = 0.25 # silence minimal (250 ms) pour détecter une pause nette
GAP_MAX = 1.5       # si deux segments sont séparés par <= 1.5 s, on peut les regrouper jusqu'à MAX_CLIP

os.makedirs(f"{WORK}/data/audio", exist_ok=True)
WHISPER_SIZE = globals().get("WHISPER_SIZE", "large-v3")
DEVICE = globals().get("DEVICE", "cuda" if torch.cuda.is_available() else "cpu")
COMPUTE = globals().get("COMPUTE", "float16" if DEVICE == "cuda" else "int8")
model = WhisperModel(WHISPER_SIZE, device=DEVICE, compute_type=COMPUTE)

CACHE = {}
def load_full(path):
    try:
        data, sr = sf.read(path, dtype="float32")
    except Exception:
        import subprocess
        tmp = "/tmp/_decode.wav"
        subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", path, "-ac", "1", "-ar", str(TARGET_SR), tmp], check=True)
        data, sr = sf.read(tmp, dtype="float32")
    if data.ndim > 1:
        data = data.mean(axis=1)
    if sr != TARGET_SR:
        data = torchaudio.functional.resample(torch.from_numpy(data), sr, TARGET_SR).numpy()
    return data, TARGET_SR

def get_audio(path):
    if path not in CACHE:
        CACHE.clear()
        try:
            CACHE[path] = load_full(path)
        except Exception as ex:
            print("skip", path, "->", ex)
            CACHE[path] = None
    return CACHE[path]

def split_audio_by_silences(audio, sr, silence_db=SILENCE_DB, min_silence_s=MIN_SILENCE_S):
    """
    1. Détecte tous les vrais silences (RMS < seuil) dans l'audio.
    2. Découpe uniquement au milieu des silences les plus longs.
    3. Regroupe les petits morceaux jusqu'à MAX_CLIP, et redécoupe les morceaux > MAX_CLIP
       sur leur pause interne la plus calme.
    """
    hop = int(0.01 * sr)  # pas de 10 ms
    win = int(0.02 * sr)  # fenêtre de 20 ms
    n_frames = max(1, (len(audio) - win) // hop + 1)
    sq = audio * audio
    csum = np.cumsum(np.insert(sq, 0, 0.0))
    starts = np.arange(n_frames) * hop
    rms = np.sqrt(np.maximum(1e-12, (csum[starts + win] - csum[starts]) / win))
    # Référence robuste (95e percentile pour ignorer un clic isolé)
    ref_rms = np.percentile(rms, 95)
    thresh = ref_rms * (10.0 ** (silence_db / 20.0))
    is_speech = rms > thresh

    # Extraire les intervalles de parole contigus
    min_sil_frames = int(min_silence_s / 0.01)
    speech_segs = []
    in_sp = False
    sp_start = 0
    sil_count = 0
    for idx, sp in enumerate(is_speech):
        if sp:
            if not in_sp:
                sp_start = idx
                in_sp = True
            sil_count = 0
        else:
            if in_sp:
                sil_count += 1
                if sil_count >= min_sil_frames:
                    sp_end = idx - sil_count + 1
                    speech_segs.append([sp_start * hop, min(len(audio), sp_end * hop + win)])
                    in_sp = False
                    sil_count = 0
    if in_sp:
        speech_segs.append([sp_start * hop, len(audio)])

    # Si un segment dépasse MAX_CLIP, on le redécoupe sur son creux RMS le plus calme
    def refine_long_segment(s_idx, e_idx, depth=0):
        dur = (e_idx - s_idx) / sr
        if dur <= MAX_CLIP:
            return [(s_idx, e_idx)]
        if depth > 4:
            return []
        # Chercher la pause la plus calme dans le tiers central [25% .. 75%]
        margin = int(0.25 * (e_idx - s_idx))
        search_a = s_idx + margin
        search_b = e_idx - margin
        f_a = search_a // hop
        f_b = min(len(rms), search_b // hop)
        if f_b <= f_a:
            return []
        # Lissage sur 120 ms pour trouver une vraie pause (pas un creux entre 2 voyelles)
        k = 12
        sub = rms[f_a:f_b]
        if len(sub) <= k:
            return []
        smooth = np.convolve(sub, np.ones(k) / k, mode="valid")
        best_f = f_a + int(np.argmin(smooth)) + k // 2
        # Ne couper que si ce creux est vraiment calme (< 2.5x seuil)
        if rms[best_f] > thresh * 2.5:
            return []
        cut = best_f * hop
        return refine_long_segment(s_idx, cut, depth + 1) + refine_long_segment(cut, e_idx, depth + 1)

    refined = []
    for s_idx, e_idx in speech_segs:
        refined.extend(refine_long_segment(s_idx, e_idx))

    # Regrouper les petits segments séparés par <= GAP_MAX tant que total <= MAX_CLIP
    pad = int(0.12 * sr)  # 120 ms de silence propre de chaque côté
    merged = []
    cur_s, cur_e = None, None
    for s_idx, e_idx in refined:
        if cur_s is None:
            cur_s, cur_e = s_idx, e_idx
        elif (s_idx - cur_e) / sr <= GAP_MAX and (e_idx - cur_s) / sr <= MAX_CLIP:
            cur_e = e_idx
        else:
            if (cur_e - cur_s) / sr >= MIN_CLIP:
                merged.append((max(0, cur_s - pad), min(len(audio), cur_e + pad)))
            cur_s, cur_e = s_idx, e_idx
    if cur_s is not None and (cur_e - cur_s) / sr >= MIN_CLIP:
        merged.append((max(0, cur_s - pad), min(len(audio), cur_e + pad)))
    return merged

# --- 1. Découpe sur les silences, puis 2. Transcription Whisper par clip ---
rows, durs = {}, {}
n_cand = 0
fade_len = int(0.01 * TARGET_SR)

for f in FILES:
    res = get_audio(f)
    if res is None:
        continue
    audio, sr = res
    clips_bounds = split_audio_by_silences(audio, sr)
    n_cand += len(clips_bounds)
    print(f"{os.path.basename(f)} -> {len(clips_bounds)} segments détectés sur silences, transcription...")

    for s_idx, e_idx in clips_bounds:
        clip = audio[s_idx:e_idx].copy()
        if len(clip) > 2 * fade_len:
            clip[:fade_len] *= np.linspace(0.0, 1.0, fade_len, dtype=np.float32)
            clip[-fade_len:] *= np.linspace(1.0, 0.0, fade_len, dtype=np.float32)

        wav16 = torchaudio.functional.resample(
            torch.from_numpy(clip), TARGET_SR, WHISPER_SR
        ).numpy().astype(np.float32)
        segs, _ = model.transcribe(wav16, language="fr", vad_filter=False, beam_size=5)
        seg_list = [s for s in segs if s.text.strip()]
        if not seg_list:
            continue
        tx = " ".join(s.text.strip() for s in seg_list)
        lp = min(s.avg_logprob for s in seg_list)
        if len(tx) < 15:
            continue

        cid = f"clip_{len(rows):04d}"
        out = f"{WORK}/data/audio/{cid}.wav"
        sf.write(out, clip, sr, subtype="PCM_16")
        rows[cid] = {"id": cid, "audio_path": out, "text": tx,
                     "language_id": "fr", "logprob": lp}
        durs[cid] = len(clip) / sr

if not rows:
    raise RuntimeError("Aucun clip exploitable produit. Vérifie que tes fichiers audio sont lisibles (WAV conseillés).")

print(f"\n{n_cand} segments sur silences, {len(rows)} clips retenus. Durée totale : {sum(durs.values()) / 60:.1f} min")

print("=== Écoute : repère l'ID d'un clip 100% narrateur (pour la section 7) ===")
for r in sorted(rows.values(), key=lambda r: -durs[r["id"]])[:10]:
    print(f"\n{r['id']} ({durs[r['id']]:.1f}s) :", r["text"][:90])
    ipd.display(ipd.Audio(r["audio_path"]))


## 6. Garder uniquement le narrateur + les meilleures parties
1. **Locuteur** (si `NARRATOR_CLIP_ID` est rempli) : similarité d'empreinte vocale avec le clip de référence ; `SIM_THRESHOLD` décide (0.65 tolérant, 0.80 strict).
2. **Qualité** : les clips dont la transcription est peu fiable sont jetés (parole hachée, musique résiduelle).

Résultat : manifestes train (90 %) / dev (10 %) + voix de référence. La cellule est relançable.

In [ ]:
import os, json, random, shutil, sys, subprocess
import numpy as np
import soundfile as sf

# <-- REMPLIS avec l'ID d'un clip 100% narrateur écouté à la section 6 (ex: "clip_0042")
# Laisse vide si ton audio ne contient que le narrateur (ex: enregistrements Audacity).
NARRATOR_CLIP_ID = ""
MIN_LOGPROB = -0.45  # en dessous : parole peu claire => on jette

WORK = "/content/work"

# --- Remise en forme de rows : cette cellule est relançable autant de fois que nécessaire ---
rows = globals().get("rows", [])
if isinstance(rows, dict):
    rows = list(rows.values())
if rows and isinstance(rows[0], str):
    # rows contient des lignes JSON en texte : on les redécode en dictionnaires
    rows = [json.loads(s) for s in rows if s.strip()]
if not rows:
    # dernier recours : recharger les manifestes écrits lors d'un run précédent
    for name in ("train", "dev"):
        p = f"{WORK}/data/{name}.jsonl"
        if os.path.exists(p):
            with open(p, encoding="utf-8") as fh:
                rows += [json.loads(l) for l in fh if l.strip()]
assert rows and isinstance(rows[0], dict), (
    "rows est vide ou invalide : relance d'abord la cellule 6 (transcription + découpe), puis celle-ci"
)
print(f"{len(rows)} clips chargés")

# --- durs incomplet : durée réelle des clips relue sur le disque ---
durs = globals().get("durs", {})
for r in rows:
    if r["id"] not in durs:
        info = sf.info(r["audio_path"])
        durs[r["id"]] = info.frames / info.samplerate

have_logprob = all("logprob" in r for r in rows)

if NARRATOR_CLIP_ID:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "resemblyzer"], check=True)
    from resemblyzer import VoiceEncoder, preprocess_wav
    encoder = VoiceEncoder(device="cuda" if globals().get("HAS_GPU", True) else "cpu")
    embs = {}
    for r in rows:
        embs[r["id"]] = encoder.embed_utterance(preprocess_wav(r["audio_path"]))
    ref_emb = embs[NARRATOR_CLIP_ID]
    norm = np.linalg.norm(ref_emb)
    sims = {cid: float(np.dot(e, ref_emb) / (np.linalg.norm(e) * norm)) for cid, e in embs.items()}
    print("Répartition des similarités avec le clip narrateur :")
    for th in (0.60, 0.65, 0.70, 0.75, 0.80, 0.85):
        print(f"  similarité >= {th} : {sum(1 for s in sims.values() if s >= th)} clips")
    before = len(rows)
    rows = [r for r in rows if sims[r["id"]] >= SIM_THRESHOLD]
    print(f"Filtre narrateur : {len(rows)} / {before} clips gardés (SIM_THRESHOLD = {SIM_THRESHOLD})")
    if len(rows) < 20:
        print("Peu de clips : écoute le clip de référence, puis baisse SIM_THRESHOLD (ex: 0.70) et relance.")
else:
    print("NARRATOR_CLIP_ID vide : pas de filtrage locuteur (OK si l'audio ne contient que le narrateur)")

# --- Filtre qualité : ne garder que les meilleures parties ---
if have_logprob:
    before = len(rows)
    rows = [r for r in rows if r["logprob"] >= MIN_LOGPROB]
    print(f"Filtre qualité (logprob >= {MIN_LOGPROB}) : {len(rows)} / {before} clips gardés")
else:
    print("Score logprob absent : filtre qualité sauté (dans ce cas, la qualité a déjà été assurée en amont)")
if len(rows) < 10:
    raise RuntimeError(
        "Trop peu de clips après filtrage. Relâche SIM_THRESHOLD ou MIN_LOGPROB, ou ajoute des données."
    )

# --- Split 90/10 + manifestes + voix de référence ---
rows = [dict(r) for r in rows]
random.seed(globals().get("SPLIT_SEED", 42))
random.shuffle(rows)
n_dev = max(1, int(len(rows) * globals().get("DEV_RATIO", 0.10)))
dev, train = rows[:n_dev], rows[n_dev:]

for name, part in [("train", train), ("dev", dev)]:
    with open(f"{WORK}/data/{name}.jsonl", "w", encoding="utf-8") as fh:
        for r in part:
            m = {k: r[k] for k in ("id", "audio_path", "text", "language_id")}
            fh.write(json.dumps(m, ensure_ascii=False) + "\n")

if NARRATOR_CLIP_ID and any(r["id"] == NARRATOR_CLIP_ID for r in rows):
    ref = next(r for r in rows if r["id"] == NARRATOR_CLIP_ID)
else:
    cands = [r for r in train if 5.0 <= durs.get(r["id"], 0) <= 12.0]
    ref = max(cands, key=lambda r: len(r["text"])) if cands else max(train, key=lambda r: len(r["text"]))
shutil.copy(ref["audio_path"], f"{WORK}/ref_narrateur.wav")
with open(f"{WORK}/ref_narrateur.txt", "w", encoding="utf-8") as fh:
    fh.write(ref["text"])

print(f"{len(train)} clips train / {len(dev)} clips dev")
print(f"Durée totale train : {sum(durs[r['id']] for r in train) / 60:.1f} min")
print("Voix de référence sauvegardée :", ref["text"][:120])

## 7. Export du dataset
Regroupe clips, manifestes et voix de référence dans `data/`, puis crée `dataset_narrateur.zip`.

In [ ]:
import json
import os, shutil
WORK = "/content/work"

for p in ["data/train.jsonl", "data/dev.jsonl", "ref_narrateur.wav", "ref_narrateur.txt"]:
    assert os.path.exists(f"{WORK}/{p}"), f"{p} manquant : relance les sections 5 et 6"

# la voix de référence voyage avec le dataset
shutil.copy(f"{WORK}/ref_narrateur.wav", f"{WORK}/data/ref_narrateur.wav")
shutil.copy(f"{WORK}/ref_narrateur.txt", f"{WORK}/data/ref_narrateur.txt")
n_train = sum(1 for _ in open(f"{WORK}/data/train.jsonl", encoding="utf-8"))
n_dev = sum(1 for _ in open(f"{WORK}/data/dev.jsonl", encoding="utf-8"))
meta = {
    "split_seed": globals().get("SPLIT_SEED", 42),
    "dev_ratio": globals().get("DEV_RATIO", 0.10),
    "sim_threshold": globals().get("SIM_THRESHOLD", None),
    "n_train": n_train,
    "n_dev": n_dev,
}
with open(f"{WORK}/data/meta.json", "w", encoding="utf-8") as fh:
    json.dump(meta, fh, indent=2, ensure_ascii=False)
print("meta.json :", meta)

shutil.make_archive(f"{WORK}/dataset_narrateur", "zip", WORK, "data")
zip_path = f"{WORK}/dataset_narrateur.zip"

size_mb = os.path.getsize(f"{WORK}/dataset_narrateur.zip") / 1e6
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
shutil.copy(zip_path, f"{DRIVE_OUTPUT}/dataset_narrateur.zip")
print(f"dataset_narrateur.zip créé ({size_mb:.0f} Mo)")
print(f"Copié dans ton Drive : {DRIVE_OUTPUT}/dataset_narrateur.zip")

## Prochaines étapes

`dataset_narrateur.zip` est dans ton Drive (**Mon Drive/omnivoice/**).

Pour l'entraînement (notebook 2/2), deux options :
- tu téléverses ce zip en **nouvelle version** de ton dataset Kaggle `priv375/dataset-narrateur` (Upload → New Version),
  et la suite reprend où elle était ;
- ou tu annonces dans la conversation que le zip est prêt, et la suite est enchaînée automatiquement.